# PyTorch API 与 Tensor 基础

这个 notebook 不是在讲某一个具体算法，而是在补神经网络学习最容易卡住的基础：张量、形状、视图、广播、矩阵运算。后面几乎所有深度学习模型，本质上都在对 Tensor 做这些操作。

## 读这份 notebook 时建议抓住三条主线
- Tensor 是什么：可以把它理解成“带梯度能力、可运行在 GPU 上的多维数组”。
- 形状为什么重要：网络层能不能接起来，很多时候取决于输入输出维度是否匹配。
- 运算分成哪两类：逐元素运算和矩阵运算，它们在含义和结果形状上完全不同。


## 1. Tensor 的角色

PyTorch 里最核心的数据结构就是 Tensor。它和 NumPy 数组很像，但多了三件对深度学习很重要的能力：
- 可以参与自动求导，支持反向传播。
- 可以放到 GPU 上加速计算。
- 和神经网络层、优化器、损失函数天然配合。

所以这里虽然看起来只是“数组练习”，其实是在学习神经网络的底层语言。


In [4]:
from torch import nn
import torch
import numpy as np
from matplotlib import pyplot as plt

plt.rcParams['font.sans-serif'] = ['SimHei']  # 设置中文字体为黑体
plt.rcParams['axes.unicode_minus'] = False  # 解决负号显示问题

## 2. 创建 Tensor 的几种方式

这一段在帮助你建立“如何初始化数据”的直觉。
- `tensor()` 常用于把现有数据转成张量。
- `zeros/ones/full` 常用于初始化输入、掩码或占位数据。
- `randn` 常用于参数或样本的随机初始化。

学习时重点看：不同初始化方式对应的数值分布不同，这会直接影响模型训练是否稳定。


In [5]:
# 使用列表创建Tensor
# 可切片
t1 = torch.tensor([1, 2, 3, 4, 5])
a1 = np.array([3, 4, 5])
t2 = torch.tensor(a1)

print(t1)
print(t2)

tensor([1, 2, 3, 4, 5])
tensor([3, 4, 5])


In [7]:
# 填充
print(torch.empty(3, 4))  # 创建一个3行4列的空Tensor
print(torch.zeros(3, 4))  # 创建一个3行4列的全0 Tensor
print(torch.ones(3, 4))  # 创建一个3行4列的全1 Tensor
print(torch.full((3, 4), 5))  # 创建一个3行4列的全5 Tensor
print(torch.eye(3))  # 创建一个3行3列的单位矩阵
print(torch.randn(3, 4))  # 创建一个3行4列的随机Tensor，元素服从标准正态分布

tensor([[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]])
tensor([[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]])
tensor([[1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]])
tensor([[5, 5, 5, 5],
        [5, 5, 5, 5],
        [5, 5, 5, 5]])
tensor([[1., 0., 0.],
        [0., 1., 0.],
        [0., 0., 1.]])
tensor([[ 1.3462,  0.1005,  0.0626, -0.9453],
        [ 0.2213,  0.9856,  0.9168,  0.7850],
        [-0.7757,  1.8029,  0.3963, -2.2191]])


## 3. `tensor` 和 `Tensor` 的区别

这是初学 PyTorch 很容易混淆的一点：
- `torch.tensor(...)` 更像工厂函数，按给定数据创建张量。
- `torch.Tensor(...)` 是张量类型本身，行为上更接近类构造入口。

实际开发里更推荐优先使用 `torch.tensor(...)`，因为它的语义更清晰，也更不容易踩默认类型相关的坑。


In [25]:
# tensor 和 Tensor 的区别
# tensor是一个函数，可以用来创建Tensor对象，而Tensor是一个类，表示一个多维数组。
# tensor函数是torch模块中的一个工厂函数，用于创建Tensor对象。它可以接受各种类型的输入，如列表、numpy数组等，并返回一个Tensor对象。
# Tensor类是torch模块中的一个类，表示一个多维数组。它具有各种属性和方法，可以用于进行各种操作，如索引、切片、数学运算等。

print(type(torch.Tensor))  # <class 'type'> Tensor是一个类
print(type(torch.tensor))  # <class 'function'> tensor是一个函数

t = torch.tensor(np.arange(1))
print(t)
print(t.item())  # 获取Tensor中的单个元素值 单个元素的Tensor才能使用item()方法

print(torch.Tensor([1]).item())  # 创建一个Tensor，元素为1、2、3

<class 'torch._C._TensorMeta'>
<class 'builtin_function_or_method'>
tensor([0])
0
1.0


In [23]:
t = torch.Tensor([1, 2, 3])  # 将Tensor转换为numpy数组
print(type(t))
arr = t.numpy()  # 将Tensor转换为numpy数组
print(type(arr))

print(t.size() == t.shape)  # 获取Tensor的形状
print(t.size(0))  # 获取Tensor的第0维的大小

<class 'torch.Tensor'>
<class 'numpy.ndarray'>
True
3


## 4. 视图、reshape 与内存共享

这一段特别重要，因为很多“改了 A 为什么 B 也变了”的问题都来自这里。

`reshape/view` 往往不是复制一份新数据，而是换一个角度去看同一块内存。如果两个张量共享底层数据，那么改动其中一个，另一个也可能一起变化。

这和神经网络中的特征展平、卷积输出重排、batch 维度变换关系很大。


In [38]:
arr1 = np.array([[1, 2, 3], [4, 5, 6]])
print(id(arr1))
arr2 = arr1.reshape(3, 2)
print(id(arr2))  # reshape方法返回一个新的视图，内存地址不同

print(arr1)
print(arr2)

arr2[0, 0] = 100
print(arr1)  # arr1也被修改了，因为arr2是arr1的视图, 它们共享相同的数据
print(arr2)
print('---' * 10)

t1 = torch.tensor([[1, 2, 3], [4, 5, 6]])
print(id(t1))
# view 方法与 reshape 方法类似，但 view 方法只能在满足一定条件的情况下使用，例如输入Tensor必须是连续的（contiguous）。
# 如果输入Tensor不是连续的，view 方法会抛出一个错误。
# -1表示自动计算维度的大小，view方法会根据输入Tensor的总元素数量和其他指定的维度来计算出-1所在维度的大小。
t2 = t1.view([3, -1])  # view方法返回一个新的视图，内存地址不同, 但它们共享相同的数据
print(id(t2))
print(t1)
print(t2)
t2[0, 0] = 100
print(t1)  # t1也被修改了，因为t2是t1的视图
print(t2)

2010375441360
2010375447120
[[1 2 3]
 [4 5 6]]
[[1 2]
 [3 4]
 [5 6]]
[[100   2   3]
 [  4   5   6]]
[[100   2]
 [  3   4]
 [  5   6]]
------------------------------
2010286402656
2010374382816
tensor([[1, 2, 3],
        [4, 5, 6]])
tensor([[1, 2],
        [3, 4],
        [5, 6]])
tensor([[100,   2,   3],
        [  4,   5,   6]])
tensor([[100,   2],
        [  3,   4],
        [  5,   6]])


## 5. 随机数与统计运算

深度学习训练离不开随机性：
- 参数初始化常用随机数。
- mini-batch 顺序常带有随机打乱。
- dropout 本身就是随机失活。

而 `max/min/mean/sum` 这些统计操作则经常出现在损失计算、日志记录和特征汇总里。


In [52]:
# randn方法创建一个随机Tensor，元素服从标准正态分布
# rand方法创建一个随机Tensor，元素服从均匀分布
# randint方法创建一个随机Tensor，元素为指定范围内的整数
# randperm方法创建一个随机Tensor，元素为指定范围内的整数，并且不重复
# randint_like方法创建一个随机Tensor，元素为指定范围内的整数，形状与输入Tensor相同

t = torch.randn(3, 4)
print(t)
print(t.dim())  # 获取Tensor的维度数量
print(t.max())  # 获取Tensor中的最大值
print(t.min())  # 获取Tensor中的最小值
print(t.mean())  # 获取Tensor的平均值
print(t.sum())  # 获取Tensor的元素总和

print((t.t()))  # 转置

tensor([[ 0.6174, -0.3520, -0.2766, -0.2209],
        [-0.2005, -0.3618, -0.7090, -1.0655],
        [ 0.9371, -0.3403,  1.4149, -0.6977]])
2
tensor(1.4149)
tensor(-1.0655)
tensor(-0.1046)
tensor(-1.2550)
tensor([[ 0.6174, -0.2005,  0.9371],
        [-0.3520, -0.3618, -0.3403],
        [-0.2766, -0.7090,  1.4149],
        [-0.2209, -1.0655, -0.6977]])


## 6. 转置与维度重排

`transpose` 和 `permute` 都是在交换维度顺序，但用途不同：
- `transpose` 更适合交换两个维度。
- `permute` 更适合一次性重排多个维度。

图像数据经常需要在 `batch, channel, height, width` 之间调整顺序，所以这部分是后续 CNN 的前置基础。


In [57]:
# permute方法可以指定任意维度的顺序进行转置，而transpose方法只能交换两个维度。
# permute类似numpy的rollaxis方法，可以指定任意维度的顺序进行转置比rollaxis更方便灵活。
t = torch.tensor(np.arange(24).reshape(2, 3, 4))
print(t.shape)  # 获取Tensor的形状

print(t.transpose(0, 1).shape)  # 转置指定维度
print(t.permute(1, 0, 2).shape)  # 转置指定维度
print(t.permute(2, 1, 0).shape)  # 转置指定维度

torch.Size([2, 3, 4])
torch.Size([3, 2, 4])
torch.Size([3, 2, 4])
torch.Size([4, 3, 2])


## 7. 原地操作、逐元素运算与矩阵乘法

这里在区分三件很容易混在一起的事：
- 原地操作：直接改原变量，速度快但更容易影响梯度和调试。
- 逐元素运算：形状通常相同，元素一一对应计算。
- 矩阵乘法：在特征维度上做线性组合，是神经网络全连接层的核心。

看到 `@` 时要马上意识到：这不再是普通乘法，而是在做线性代数意义上的变换。


In [61]:
# 方法后加下划线表示原地操作，会直接修改原来的Tensor，而不返回一个新的Tensor。

x = torch.tensor(np.arange(12).reshape(3, 4), dtype=torch.int8)  # 指定数据类型为int8
print(x)
y = torch.ones_like(x)  # 创建一个与x形状相同的全1 Tensor
print(y)

print(x - y)  # x的每个元素减去y的对应元素，返回一个新的Tensor

x.sub_(y)  # 原地操作，x的每个元素减去y的对应元素
print(x)

tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]], dtype=torch.int8)
tensor([[1, 1, 1, 1],
        [1, 1, 1, 1],
        [1, 1, 1, 1]], dtype=torch.int8)
tensor([[-1,  0,  1,  2],
        [ 3,  4,  5,  6],
        [ 7,  8,  9, 10]], dtype=torch.int8)
tensor([[-1,  0,  1,  2],
        [ 3,  4,  5,  6],
        [ 7,  8,  9, 10]], dtype=torch.int8)


In [59]:
t1 = torch.tensor(np.arange(16).reshape(16, 1), dtype=torch.float32)
t2 = torch.tensor(np.arange(16, 32).reshape(16, 1), dtype=torch.float32)

print(((t1 - t2) ** 2).mean())  # 计算均方误差

tensor(256.)


In [70]:
x = torch.tensor([[1, 2], [3, 4], [5, 6]])
y = torch.tensor([[5, 6], [7, 8], [9, 10]])

print(x * y)  # x的每个元素乘以y的对应元素，返回一个新的Tensor
# x.mul_(y)  # 原地操作，x的每个元素乘以y的对应元素

print(x @ y.t())  # x与y的转置进行矩阵乘法，返回一个新的Tensor

tensor([[ 5, 12],
        [21, 32],
        [45, 60]])
tensor([[ 17,  23,  29],
        [ 39,  53,  67],
        [ 61,  83, 105]])
tensor([[1, 2],
        [3, 0],
        [1, 2]])
